In [ ]:
import warnings
import numpy as np
import pandas as pd
import seaborn as sb
from sklearn import svm, tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import RidgeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")

data = pd.read_csv("data/ortodoncja.csv")

In [ ]:
# data = data.drop(["9_SN/MP", "9_Facial axis", "9_Y-axis"], axis="columns")
# data = data.drop("9_SNPog", axis="columns")
# data = data.drop("9_SNB", axis="columns")
# data = data.drop("9_SNA", axis="columns")
data = data.drop("9_Mn Ramus Angle", axis="columns")
# data = data.drop("9_Mn Base angle", axis="columns")
# data = data.drop("9_AFH:PFH", axis="columns")
# data = data.drop(["12_SN/MP", "12_Facial axis", "12_Y-axis"], axis="columns")
# data = data.drop("12_SNPog", axis="columns")
# data = data.drop("12_SNB", axis="columns")
# data = data.drop("12_SNA", axis="columns")
data = data.drop("12_Mn Ramus Angle", axis="columns")
# data = data.drop("12_Mn Base angle", axis="columns")
# data = data.drop("12_AFH:PFH", axis="columns")

In [ ]:
X = data.drop("growth direction", axis="columns")
X = pd.DataFrame(StandardScaler().fit_transform(X))
y = data["growth direction"]

In [ ]:
# growth_direction_to_numbers = {"horizontal": 1, "normal": 0, "vertical": -1}
#
# X["numeric_growth"] = [growth_direction_to_numbers[growth] for growth in y]

In [ ]:
# data.groupby("growth direction")["growth direction"].count()
sb.heatmap(X.corr())

In [ ]:
SEED = 2137
ITERATIONS = 10

In [ ]:
classifiers = [
    ("GaussianNB", GaussianNB),
    ("Ridge", RidgeClassifier),
    ("NeuralNetwork", MLPClassifier),
    ("LinearSVC", svm.LinearSVC),
    ("SVC", svm.SVC),
    ("KNN", KNeighborsClassifier),
    ("RandomForest", RandomForestClassifier),
    ("HistGB", HistGradientBoostingClassifier),
    ("DecisionTree", tree.DecisionTreeClassifier),
]

results = {}
for name, ClfClass in classifiers:
    acc_scores = []
    for i in range(ITERATIONS):
        smote = SMOTE(sampling_strategy='minority', random_state=SEED + i)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED + i)
        X_train_SMOTE, y_train_SMOTE = smote.fit_resample(X_train, y_train)

        kwargs = {}
        if "random_state" in ClfClass().get_params():
            kwargs["random_state"] = SEED + i
        clf = ClfClass(**kwargs)

        clf.fit(X_train_SMOTE, y_train_SMOTE)
        y_pred = clf.predict(X_test)
        acc_scores.append(accuracy_score(y_test, y_pred))

    results[name] = np.array(acc_scores)
    print(f"{name:<13} mean: {results[name].mean():10.5f}  std: {results[name].std():10.5f}")
